# KOBIS 월간(7월) 누적 박스오피스 통계 (Google Colab)

**목표**: KOBIS는 '월간' API가 따로 없어서, 일별 API를 31번 반복 호출한 뒤 영화별로 관객수를 계속 더해서 직접 '월간 집계'를 만듭니다.

## 실습 흐름
1. 설정 + 31일치 반복 호출 + 집계 + TOP10 출력 + CSV 저장까지 한 셀에서 진행

> ⚠️ **에러가 나면 셀을 하나씩 다시 실행하지 말고, 메뉴에서 "런타임 → 모두 실행(Run all)"을 눌러 처음부터 다시 실행하세요.**
> (중간 셀만 다시 실행하면 이전 실행 결과가 남아있어 오류 원인 파악이 더 어려워집니다.)

이 CSV(`boxoffice_2026_07_monthly.csv`)는 다음 실습인 **자동 팩트체킹(week3_factcheck_demo_colab.ipynb)**에서 '정답 데이터'로 사용되니, 실행 후 꼭 다운로드해두세요.


## 셀 1. 설정 + 31일치 반복 수집 + 집계 + 저장

In [ ]:
import requests
import csv
import time
from datetime import datetime, timedelta

KOBIS_KEY = "여기에_발급받은_인증키_입력"
url = "http://kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchDailyBoxOfficeList.json"

# 1) 7월 1일부터 31일까지 날짜 목록 만들기
start_date = datetime(2026, 7, 1)
end_date = datetime(2026, 7, 31)

date_list = []
current = start_date
while current <= end_date:
    date_list.append(current.strftime("%Y%m%d"))
    current += timedelta(days=1)

print(f"조회할 날짜 수: {len(date_list)}일\n")

# 2) 영화별 누적 관객수/매출액을 저장할 딕셔너리 준비
#    형태: {"영화제목": {"audience": 누적관객수, "sales": 누적매출액}}
movie_totals = {}

# 3) 날짜 하나씩(31번) 반복하면서 그날의 박스오피스(10개)를 또 반복 처리
for i, date_str in enumerate(date_list):
    params = {"key": KOBIS_KEY, "targetDt": date_str}
    response = requests.get(url, params=params)
    data = response.json()
    daily_list = data["boxOfficeResult"]["dailyBoxOfficeList"]

    for movie in daily_list:
        title = movie["movieNm"]
        audience = int(movie["audiCnt"])
        sales = int(movie["salesAmt"])
        if title in movie_totals:
            movie_totals[title]["audience"] += audience
            movie_totals[title]["sales"] += sales
        else:
            movie_totals[title] = {"audience": audience, "sales": sales}

    time.sleep(0.2)  # API 서버에 너무 빠르게 연속 요청하지 않도록(매너 있는 API 사용)
    print(f"[{i+1}/{len(date_list)}] {date_str} 처리 완료")

# 4) 전체 누적 관객수/매출액 합계
total_audience = sum(v["audience"] for v in movie_totals.values())
total_sales = sum(v["sales"] for v in movie_totals.values())

# 5) 관객수 많은 순 TOP10
top10 = sorted(movie_totals.items(), key=lambda x: x[1]["audience"], reverse=True)[:10]

print(f"\n=== 2026년 7월 총 누적 관객수: {total_audience:,}명 / 총 매출액: {total_sales:,}원 ===\n")
print("=== 7월 박스오피스 TOP 10 (월간 누적 기준) ===\n")
for rank, (title, stats) in enumerate(top10, start=1):
    print(f"{rank}위 | {title} | 관객수 {stats['audience']:,}명 | 매출액 {stats['sales']:,}원")

# 6) CSV로 저장
csv_filename = "boxoffice_2026_07_monthly.csv"
with open(csv_filename, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["순위", "영화제목", "7월 누적관객수", "7월 누적매출액"])
    for rank, (title, stats) in enumerate(top10, start=1):
        writer.writerow([rank, title, stats["audience"], stats["sales"]])

print(f"\n저장 완료: {csv_filename}")
print("이 파일은 왼쪽 폴더 아이콘에서 다운로드해서, 다음 실습(자동 팩트체킹)에서 업로드해야 합니다!")

# from google.colab import files
# files.download(csv_filename)
